In [11]:
import pandas as pd
import yfinance as yf
import numpy as np
from scipy.stats import norm

## Stock Risk Analysis

The goal of this project is to analyze the risk of holding a group of investment, specifically nine U.S stocks (AAPL, GOOG, COST, CVX, JNJ, MCD, NVDA) and one goal etf (GLD) using a 10-year data from Auguest 29, 2016 to August 28, 2028.

## Step 1: Download Data from Yahoo Finance

In [12]:
tickers = ["AAPL", "GOOG", "COST", "CVX", "JNJ", "JPM", "MCD", "NVDA", "GLD"]

df = yf.download(tickers, start="2016-08-29", end="2026-08-28")

close_prices = df["Close"]

# Piece 1: log returns + drop first NaN row
log_returns = np.log(close_prices / close_prices.shift(1))
log_returns = log_returns.dropna()   # <- method to drop NaN rows

# Piece 2: build summary table
summary = pd.concat([log_returns.mean(), log_returns.std(), log_returns.skew()], axis=1)
summary = summary.rename(columns={0: 'Mean', 1: 'Standard Deviation', 2: 'Skew'})

# Piece 3: annualize mean and std columns only
summary['Mean'] = summary['Mean'] * 252
summary['Standard Deviation'] = summary['Standard Deviation'] * np.sqrt(252)

print(summary)

[*********************100%***********************]  9 of 9 completed

            Mean  Standard Deviation      Skew
Ticker                                        
AAPL    0.256364            0.291216 -0.114652
COST    0.191521            0.221379 -0.533030
CVX     0.109695            0.295028 -1.090142
GLD     0.121161            0.163233 -0.719254
GOOG    0.218445            0.294283 -0.111309
JNJ     0.107112            0.188385 -0.403720
JPM     0.193488            0.273031 -0.092009
MCD     0.105671            0.205393 -0.238372
NVDA    0.502486            0.497605  0.079450


## Stage 2 Self-Check: Which stock is riskiest?

**By volatility alone (annualized std dev):**
NVDA is by far the most volatile (49.8%), followed by CVX, GOOG, and AAPL 
in the 29-30% range. GLD is the calmest by a wide margin (16.4%).

**But volatility alone is misleading.**
Checking mean vs. std across all 9 tickers: every single stock has mean < std.
This is a general feature of daily returns (short-term noise dwarfs long-term 
drift) — not something that distinguishes any one stock from the rest.

**Skew reveals a different risk story.**
GLD has the lowest volatility (16.4%) but a strongly negative skew (-0.725) —
almost as negative as CVX (-1.092), the single most negative in the basket.
Compare this to JPM: nearly double GLD's volatility (27.3%) but a much more 
symmetric skew (-0.092).

**Conclusion:** GLD is the calmest asset day-to-day, but its negative skew means 
its rare "bad days" are disproportionately severe relative to its normal behavior — 
a risk that standard deviation alone doesn't capture. "Riskiest" depends on the 
definition: lowest day-to-day volatility (GLD) is not the same as lowest exposure 
to a severe, asymmetric crash (arguably JPM, given its much more symmetric tail 
despite higher volatility).

## Estimate the Probability of a >%5 Single-Day Drop

In [13]:
summary['Mean'] = summary['Mean'] / 252
summary['Standard Deviation'] = summary['Standard Deviation'] / np.sqrt(252)
summary["z score"] = (-0.05 - summary['Mean'])/summary['Standard Deviation']
print(summary)

            Mean  Standard Deviation      Skew   z score
Ticker                                                  
AAPL    0.001017            0.018345 -0.114652 -2.781014
COST    0.000760            0.013946 -0.533030 -3.639863
CVX     0.000435            0.018585 -1.090142 -2.713757
GLD     0.000481            0.010283 -0.719254 -4.909297
GOOG    0.000867            0.018538 -0.111309 -2.743909
JNJ     0.000425            0.011867 -0.403720 -4.249123
JPM     0.000768            0.017199 -0.092009 -2.951732
MCD     0.000419            0.012939 -0.238372 -3.896832
NVDA    0.001994            0.031346  0.079450 -1.658704


In [20]:
summary["Probability 5% Drop"] = norm.cdf(summary["z score"]) * 100
print(summary)

            Mean  Standard Deviation      Skew   z score  Probability 5% Drop
Ticker                                                                       
AAPL    0.001017            0.018345 -0.114652 -2.781014             0.270947
COST    0.000760            0.013946 -0.533030 -3.639863             0.013639
CVX     0.000435            0.018585 -1.090142 -2.713757             0.332625
GLD     0.000481            0.010283 -0.719254 -4.909297             0.000046
GOOG    0.000867            0.018538 -0.111309 -2.743909             0.303562
JNJ     0.000425            0.011867 -0.403720 -4.249123             0.001073
JPM     0.000768            0.017199 -0.092009 -2.951732             0.157999
MCD     0.000419            0.012939 -0.238372 -3.896832             0.004873
NVDA    0.001994            0.031346  0.079450 -1.658704             4.858774
